# Restricted-input method reference
Source: analysis/intra-park/agg_attributes.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

import os
# Working directory is explicitly configured in the first cell.


In [ ]:
for park in ['amk', 'jurong', 'punggol']:
    human_perceptions_hex = gpd.read_file(f"gis_files/hexagons/human_perception/{park}_hex_agg.geojson")
    rename_dict = {
        "beautiful_score_mean": "pb",
        "boring_score_mean": "pm",
        "lively_score_mean": "pl",
        "safety_score_mean": "ps",
    }
    human_perceptions_hex = human_perceptions_hex.rename(columns=rename_dict)
    #drop rows that have all NaN in the perception columns
    human_perceptions_hex = human_perceptions_hex.dropna(subset=rename_dict.values(), how='all')

    human_perceptions_hex = human_perceptions_hex[["grid_id", *rename_dict.values(), "geometry"]]
    print(len(human_perceptions_hex))

    segmented_hex = gpd.read_file(f"gis_files/hexagons/segmented/{park}_hex_agg.geojson")
    rename_dict_segmented = {
        "greenness_mean": "peg",
        "openness_mean": "peo",
        "furniture_mean": "pep"
    }
    segmented_hex = segmented_hex.rename(columns=rename_dict_segmented)
    #drop rows that have all NaN in the perception columns
    segmented_hex = segmented_hex.dropna(subset=rename_dict_segmented.values(), how='all')

    segmented_hex = segmented_hex[["grid_id", *rename_dict_segmented.values()]]
    print(len(segmented_hex))

    #merge the two geodataframes on grid_id
    merged_hex = human_perceptions_hex.merge(segmented_hex, on="grid_id", how="left")
    print(len(merged_hex))

    parkservice_hex = gpd.read_file(f"gis_files/hexagons/park_services/{park}_with_poi_counts.geojson")
    parkservice_hex.rename(columns={"count_within_500m": "peps"}, inplace=True)
    parkservice_hex = parkservice_hex[["grid_id", "peps"]]
    print(len(parkservice_hex))

    merged_hex = merged_hex.merge(parkservice_hex, on="grid_id", how="left")
    print(len(merged_hex))

    internalroads_hex = gpd.read_file(f"gis_files/hexagons/road_closeness/{park}_hex_closeness_updated.geojson")
    internalroads_hex.rename(columns={"closeness_mean": "per"}, inplace=True)
    internalroads_hex = internalroads_hex[["grid_id", "per"]]
    print(len(internalroads_hex))

    merged_hex = merged_hex.merge(internalroads_hex, on="grid_id", how="left")
    print(len(merged_hex))

    water_hex = gpd.read_file(f"gis_files/hexagons/proximity2water/parks_nearest_water_{park}.geojson")
    water_hex.rename(columns={"inv_distance_m": "pepw"}, inplace=True)
    water_hex = water_hex[["grid_id", "pepw"]]
    print(len(water_hex))

    merged_hex = merged_hex.merge(water_hex, on="grid_id", how="left")
    print(len(merged_hex))
    merged_hex.head()

    mrt_hex = gpd.read_file(f"gis_files/hexagons/nearest_mrt/parks_nearest_mrt_{park}.geojson")
    mrt_hex.rename(columns={"inverted_distance": "set"}, inplace=True)
    mrt_hex = mrt_hex[["grid_id", "set"]]
    print(len(mrt_hex))

    merged_hex = merged_hex.merge(mrt_hex, on="grid_id", how="left")
    print(len(merged_hex))

    merged_hex.to_file(f"gis_files/hexagons/agg_attributes/{park}_hex_agg_attributes.geojson", driver="GeoJSON")

In [ ]:
visits = gpd.read_file(f"gis_files/visitation/amk_duration_analysis.geojson")

visits.head()

In [ ]:
merged_park_together = pd.DataFrame()
for park in ['amk', 'jurong', 'punggol']:

    merged_hex = gpd.read_file(f"gis_files/hexagons/agg_attributes/{park}_hex_agg_attributes.geojson")
    print(len(merged_hex))
    visits = gpd.read_file(f"gis_files/visitation/{park}_duration_analysis.geojson")
    print(len(visits))
    merged_hex = merged_hex.merge(visits[['grid_id','F_h','D_h']], on="grid_id", how="left")
    mean_visits = merged_hex['F_h'].mean()
    median_visits = merged_hex['F_h'].median()
    std_visits = merged_hex['F_h'].std()
    mean_duration = merged_hex['D_h'].mean()
    median_duration = merged_hex['D_h'].median()
    std_duration = merged_hex['D_h'].std()
    print(f"{park} - Mean visits: {mean_visits}, Median visits: {median_visits}, Std visits: {std_visits}")
    print(f"{park} - Mean duration: {mean_duration}, Median duration: {median_duration}, Std duration: {std_duration}")
    print(len(merged_hex))
    merged_hex['park'] = park
    merged_hex.to_file(f"gis_files/hexagons/agg_attributes/{park}_hex_agg_attributes_visits.geojson", driver="GeoJSON")
    merged_park_together = pd.concat([merged_park_together, merged_hex])

merged_park_together.drop(columns=['pw', 'pd'], inplace=True, errors='ignore')

In [ ]:
merged_park_together.to_file("gis_files/hexagons/agg_attributes/parks_allthree_hex_agg_attributes_visits.geojson", driver="GeoJSON")

In [ ]:
import matplotlib.pyplot as plt

parks = ['amk', 'jurong', 'punggol']

# collect visit counts for all parks
counts = []
for pk in parks:
    v = gpd.read_file(f"gis_files/hexagons/visiting_counts/hex_fishnet_count_visits_{pk}.geojson")[['visit_count']].copy()
    v['park'] = pk
    counts.append(v)

counts_df = pd.concat(counts, ignore_index=True)

# plot histograms with x limited to [1, 1000]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, pk in zip(axes, parks):
    sub = counts_df.loc[counts_df['park'] == pk, 'visit_count'].astype(float)
    ax.hist(sub, bins=1000, color='steelblue', alpha=0.8, edgecolor='white')
    ax.set_title(pk.title())
    ax.set_xlabel('visit_count')
    ax.set_xlim(1, 100)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Frequency')
plt.tight_layout()
plt.show()